# 1. Generate zombie clinical reports designed for feature vectors

This notebook **works backward**: starting from the structured table `zombie-health.csv`, it writes
synthetic clinical reports whose language is deliberately *controlled*, so that simple algorithms
(dictionary lookup + n-grams) can later turn every word or expression into a small **feature vector**.

Design rules for the generated text:

1. every report is organized in **sections** (Presentation, General Examination, Negative Findings,
   Laboratory Analysis, Clinical History, Social History, Assessment);
2. every sentence states that **one** symptom or disease is **present** or **absent**, except the
   **context sentences** (General Examination and Social History), which state no finding at all;
3. diversity comes from rotating **templates** and **synonyms**, not from free writing;
4. in the default mode, a negation word always comes **before** the term it negates
   (`POST_NEGATION = True` turns on a "hard mode" that breaks this rule on purpose).

Context sentences borrow a **word** from a symptom or disease expression and use it elsewhere:
"tongue" appears in "yellow tongue" (a symptom), in "no yellow tongue" (an absent symptom), in
"the tongue is green" (a normal zombie) and in "speaks in a strange tongue" (a language). The same
word keeps different company in each context, which Notebook 2 uses to motivate context vectors.

Besides the reports, the notebook saves the **ground truth** (which term is present or absent in
which sentence, and which borrowed word appears in which context) and the **lexicon** (the "field
guide" of known expressions) used by Notebook 2.

In [1]:
import random
import re
from pathlib import Path

import pandas as pd

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

CSV_PATH = Path("../../data/zombie/zombie-health.csv")
REPORTS_DIR = Path("reports")
SENTENCES_FILE = "sentences.csv"
ANNOTATIONS_FILE = "annotations.csv"
LEXICON_FILE = "lexicon.csv"
CONTEXT_WORDS_FILE = "context_words.csv"

# Integer seed for reproducibility: same seed -> same reports.
SEED = 42

# Number of different reports written for each patient (row of the CSV).
# Raise it to produce more text for statistics.
VARIANTS_PER_PATIENT = 1

# How many absent symptoms / ruled-out diseases each report mentions (min, max).
N_ABSENT_SYMPTOMS = (1, 2)
N_RULED_OUT = (0, 1)

# How many context sentences each report has (min, max). Use (0, 0) to turn them off.
# Normal findings: a borrowed word describes a healthy zombie ("The tongue is green").
# Everyday: a borrowed word with another meaning ("speaks in a strange tongue").
N_NORMAL_FINDINGS = (1, 2)
N_EVERYDAY = (1, 2)

# Word to illustrate in class: the word (in absent symptoms and context sentences) is FOCUS_WEIGHT
# times more likely to be drawn than any other. Use FOCUS_WEIGHT = 1 for uniform sampling.
FOCUS_WORD = "tongue"
FOCUS_WEIGHT = 5

# Hard mode: also use templates where the negation comes AFTER the term,
# e.g. "Signs of paralysis were not observed."
POST_NEGATION = False

## Controlled vocabulary

Each canonical symptom / disease of the CSV has a small bank of synonyms (surface forms).
The banks were cleaned so that each surface form can be found as an **n-gram**:
no articles, no hyphens, and at most 5 tokens ("trembling on the little finger" is the longest).

The diagnosis `nothing` is **not** a disease: it is rendered by a sentence without any disease term.

In [2]:
SYMPTOM_PHRASES = {
    "paralysis": ["paralysis", "paralyzed limbs", "loss of motor control"],
    "yellow tongue": ["yellow tongue", "yellowish tongue", "yellow tinted tongue"],
    "trembling on the little finger": [
        "trembling on the little finger",
        "tremor in the little finger",
        "shaking of the pinky finger",
    ],
    "member loss": ["member loss", "limb loss", "missing limb"],
    "chest pain": ["chest pain", "pain in the chest", "thoracic discomfort"],
    "severe anger": ["severe anger", "intense anger episodes", "acute rage attacks"],
}

DISEASE_PHRASES = {
    "bacterial infection": [
        "bacterial infection",
        "infection of bacterial origin",
        "bacterial infectious process",
    ],
    "viral infection": [
        "viral infection",
        "infection of viral origin",
        "viral infectious process",
    ],
    "bite deficit": ["bite deficit", "bite related deficit", "deficit caused by bite"],
    "fights": ["fight injury", "fight related trauma", "altercation injury"],
}

PATHOGENS = ["Cyanobacterium zumbi", "Deltaretrovirus zumbi"]

# Negation cues and scope terminators (shared with Notebook 2 through the lexicon).
NEGATION_CUES = ["no", "not", "denies", "without", "absence", "negative", "free of", "ruled out", "absent"]
SCOPE_TERMINATORS = ["but", "however"]

NO_DISEASE = "nothing"

## Sentence templates

Each list holds templates of **one polarity**. `{t}` is the term; `{T}` is the term with a capital
initial letter. Templates never place an article or a verb that must agree in number with the term,
so any synonym fits any template.

| Section | Present | Absent |
|---|---|---|
| Presentation | present symptoms | — |
| Negative Findings | — | absent symptoms |
| Laboratory Analysis | pathogen found | no pathogen (term-free sentence) |
| Clinical History | previous diagnosis | no history |
| Assessment | diagnosis | ruled-out diseases |

In [3]:
SYMPTOM_PRESENT = [
    "The patient presents with {t}.",
    "Examination reveals {t}.",
    "The patient reports {t}.",
    "Findings include {t}.",
    "Signs of {t} are evident.",
]

SYMPTOM_ABSENT = [
    "No signs of {t}.",
    "The patient denies {t}.",
    "Examination reveals no {t}.",
    "The patient presents without {t}.",
    "Absence of {t}.",
    "The patient is free of {t}.",
]

# Hard mode only: the negation cue comes AFTER the term.
SYMPTOM_ABSENT_POST = [
    "Signs of {t} were not observed.",
    "{T}: absent.",
]

LAB_PRESENT = [
    "Blood analysis identified {t}.",
    "Laboratory tests isolated {t}.",
    "Blood culture is positive for {t}.",
]

LAB_ABSENT = [
    "No pathogen was isolated in the blood analysis.",
    "Blood analysis returned no relevant findings.",
    "Laboratory tests did not identify any pathogen.",
]

HISTORY_PRESENT = [
    "History of {t}.",
    "The patient has a history of {t}.",
    "Previous diagnosis of {t}.",
]

HISTORY_ABSENT = [
    "No history of {t}.",
    "The patient denies previous {t}.",
]

HISTORY_EMPTY = [
    "No previous diagnosis was recorded.",
    "Clinical history is unremarkable.",
]

DIAGNOSIS_PRESENT = [
    "Diagnosis: {t}.",
    "Clinical impression: {t}.",
    "The patient was diagnosed with {t}.",
    "Findings are consistent with {t}.",
]

DIAGNOSIS_NONE = [
    "No disease was identified.",
    "No diagnosis was established.",
]

DISEASE_RULED_OUT = [
    "No evidence of {t}.",
    "Negative for {t}.",
    "The team ruled out {t}.",
]

DISEASE_RULED_OUT_POST = [
    "{T} was ruled out.",
]

if POST_NEGATION:
    SYMPTOM_ABSENT = SYMPTOM_ABSENT + SYMPTOM_ABSENT_POST
    DISEASE_RULED_OUT = DISEASE_RULED_OUT + DISEASE_RULED_OUT_POST

## Context sentences: the same word in other company

Each context template is paired with the **borrowed word** it shares with a symptom or disease
expression. Two families:

| Family | Section | The borrowed word... | Example |
|---|---|---|---|
| Normal findings | General Examination | describes a **healthy** zombie | "The tongue is green, as expected for a zombie." |
| Everyday | Social History | has **another meaning** or refers to someone else | "The patient speaks in a strange tongue." |

A normal finding is keyed by the term it borrows from and is only used when the patient does **not**
have that term (a zombie with a yellow tongue cannot also have a healthy green one). Everyday
sentences fit any patient.

`FOCUS_WORD` makes the absent symptoms and context sentences of one word more frequent, so a class example ("tongue")
has enough occurrences in every context.

No context sentence contains a full expression of the lexicon, so a dictionary lookup finds nothing
in them; only the borrowed words overlap. Some of them even contain a negation cue that negates
nothing in the lexicon ("detects no heartbeat").

In [4]:
NORMAL_FINDINGS = {
    "paralysis": [
        ("Motor control is preserved.", "control"),
        ("Limbs move slowly, as usual for zombies.", "limbs"),
    ],
    "yellow tongue": [
        ("The tongue is green, as expected for a zombie.", "tongue"),
        ("Tongue color: green.", "tongue"),
        ("The patient shows a healthy green tongue.", "tongue"),
        ("The tongue is dry and rough, typical of the species.", "tongue"),
        ("A grey coating covers the tongue, which is normal.", "tongue"),
    ],
    "trembling on the little finger": [
        ("The little finger is stiff and grey.", "finger"),
        ("Pinky finger nails are long and dirty, which is normal.", "finger"),
    ],
    "member loss": [
        ("Each limb is firmly attached.", "limb"),
        ("Every member is in its place.", "member"),
    ],
    "chest pain": [
        ("The chest is cold and silent, as usual.", "chest"),
        ("Stethoscope on the chest detects no heartbeat, as expected.", "chest"),
        ("The patient feels little pain, as usual for zombies.", "pain"),
    ],
    "severe anger": [
        ("Mild anger when hungry, which is normal.", "anger"),
        ("Anger level is low for a zombie.", "anger"),
    ],
    "bite deficit": [
        ("Bite force is normal for the species.", "bite"),
    ],
}

EVERYDAY = [
    ("The patient speaks in a strange tongue.", "tongue"),          # tongue = language
    ("Mother tongue: Zombish.", "tongue"),
    ("The caretaker has a sharp tongue.", "tongue"),                # idiom, and someone else's
    ("The patient sticks out the tongue at the nurses.", "tongue"), # a gesture, not a finding
    ("The patient arrived wearing a yellow raincoat.", "yellow"),   # yellow, but not the tongue
    ("The patient keeps spare brains in a wooden chest.", "chest"), # chest = box
    ("Favorite snack: finger food.", "finger"),
    ("The patient is a member of the downtown horde.", "member"),   # member = of a group
    ("A video of the patient dancing went viral.", "viral"),        # viral = popular online
    ("The patient watched a pillow fight in the waiting room.", "fight"),
    ("The caretaker expressed anger over the parking fees.", "anger"),  # someone else's anger
    ("The caretaker says the patient is a pain to feed.", "pain"),  # idiom
    ("The patient escaped from animal control twice.", "control"),
    ("The caretaker received a leaflet about infection prevention.", "infection"),
]

## Read the source table

The CSV lists only the symptoms that are **present**. The **absent** symptoms and ruled-out diseases
are derived: absent symptoms are drawn from the symptoms *not* listed for that patient, and ruled-out
diseases favor diseases that share symptoms with the true diagnosis (a plausible differential).

In [5]:
patients = pd.read_csv(CSV_PATH).fillna("")

ALL_SYMPTOMS = list(SYMPTOM_PHRASES)
ALL_DISEASES = list(DISEASE_PHRASES)

def row_symptoms(row):
    return [row[f"Symptom {i}"] for i in (1, 2, 3) if row[f"Symptom {i}"]]

# Symptoms observed for each disease in the table (used to pick a plausible differential).
DISEASE_SYMPTOMS = {d: set() for d in ALL_DISEASES}
for _, row in patients.iterrows():
    if row["Diagnosis"] in DISEASE_SYMPTOMS:
        DISEASE_SYMPTOMS[row["Diagnosis"]].update(row_symptoms(row))

print(f"Read {len(patients)} patients.")
patients

Read 19 patients.


,Name,Diagnosis,Symptom 1,Symptom 2,Symptom 3,Blood Analysis,Previous Diagnosis
0,Rot Donnadd,bacterial infection,paralysis,yellow tongue,,Cyanobacterium zumbi,
1,Pid Mught,bacterial infection,yellow tongue,,,Cyanobacterium zumbi,
2,Thulk Lebbimp,bite deficit,yellow tongue,trembling on the little finger,,,bacterial infection
3,Bouvossam Damme,bite deficit,member loss,yellow tongue,trembling on the little finger,,bacterial infection
4,Pirg Zall,viral infection,chest pain,member loss,,Deltaretrovirus zumbi,
5,Nullon Rackindock,fights,member loss,severe anger,,,
6,Shor Splitturch,nothing,severe anger,,,,
7,Ger Ackeng,bite deficit,trembling on the little finger,,,,bacterial infection
8,Gleldo Shruck,bacterial infection,yellow tongue,chest pain,,Cyanobacterium zumbi,
9,Nadross Pilch,viral infection,chest pain,,,Deltaretrovirus zumbi,


## Generate the reports

Each report is a list of `(section, sentence, facts, words)` items, where `facts` records the term
mentioned in the sentence (surface form, canonical name, type, polarity) and `words` records the
borrowed word of a context sentence (word, context family). Randomness is seeded by
`(SEED, row, variant)`, so re-running reproduces exactly the same reports.

In [6]:
SECTIONS = ["Presentation", "General Examination", "Negative Findings", "Laboratory Analysis",
            "Clinical History", "Social History", "Assessment"]

def slugify(name):
    return re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")

def fill(template, term):
    return template.format(t=term, T=term[0].upper() + term[1:])

def mention(rng, templates, canonical, kind, polarity):
    """One sentence stating that a single term is present or absent."""
    if kind == "symptom":
        surface = rng.choice(SYMPTOM_PHRASES[canonical])
    elif kind == "disease":
        surface = rng.choice(DISEASE_PHRASES[canonical])
    else:
        surface = canonical
    sentence = fill(rng.choice(templates), surface)
    return sentence, [{"surface": surface, "canonical": canonical, "type": kind, "polarity": polarity}], []

def context(sentence, word, family):
    """A context sentence: no fact, only the borrowed word and the family of its context."""
    return sentence, [], [{"word": word, "context": family}]

def focus_weight(templates):
    """FOCUS_WEIGHT if any (sentence, word) template borrows FOCUS_WORD, else 1."""
    return FOCUS_WEIGHT if any(word == FOCUS_WORD for _, word in templates) else 1

def focus_weight_term(canonical):
    """FOCUS_WEIGHT if any synonym of the symptom contains FOCUS_WORD, else 1."""
    return FOCUS_WEIGHT if any(FOCUS_WORD in s.split() for s in SYMPTOM_PHRASES[canonical]) else 1

def sample_k(rng, population, k_range, weights=None):
    k = min(rng.randint(*k_range), len(population))
    chosen = []
    pool, w = list(population), list(weights) if weights else [1] * len(population)
    for _ in range(k):
        pick = rng.choices(range(len(pool)), weights=w, k=1)[0]
        chosen.append(pool.pop(pick))
        w.pop(pick)
    return chosen

def generate_report(row, index, variant):
    rng = random.Random(f"{SEED}-{index}-{variant}")
    items = []

    present = row_symptoms(row)
    for s in present:
        items.append(("Presentation", *mention(rng, SYMPTOM_PRESENT, s, "symptom", "present")))

    diagnosis = row["Diagnosis"]
    healthy = [t for t in NORMAL_FINDINGS if t not in present and t not in (diagnosis, row["Previous Diagnosis"])]
    for t in sample_k(rng, healthy, N_NORMAL_FINDINGS, [focus_weight(NORMAL_FINDINGS[t]) for t in healthy]):
        templates = NORMAL_FINDINGS[t]
        sentence, word = rng.choices(templates, weights=[focus_weight([tp]) for tp in templates])[0]
        items.append(("General Examination", *context(sentence, word, "normal finding")))

    candidates = [s for s in ALL_SYMPTOMS if s not in present]
    for s in sample_k(rng, candidates, N_ABSENT_SYMPTOMS, [focus_weight_term(s) for s in candidates]):
        items.append(("Negative Findings", *mention(rng, SYMPTOM_ABSENT, s, "symptom", "absent")))

    if row["Blood Analysis"]:
        items.append(("Laboratory Analysis", *mention(rng, LAB_PRESENT, row["Blood Analysis"], "pathogen", "present")))
    else:
        items.append(("Laboratory Analysis", rng.choice(LAB_ABSENT), [], []))

    if row["Previous Diagnosis"]:
        items.append(("Clinical History", *mention(rng, HISTORY_PRESENT, row["Previous Diagnosis"], "disease", "present")))
    elif rng.random() < 0.5:
        items.append(("Clinical History", rng.choice(HISTORY_EMPTY), [], []))
    else:
        denied = rng.choice([d for d in ALL_DISEASES if d != diagnosis])
        items.append(("Clinical History", *mention(rng, HISTORY_ABSENT, denied, "disease", "absent")))

    # The focus word's weight is shared by its templates, so the word, not each template, is favored.
    n_focus = sum(word == FOCUS_WORD for _, word in EVERYDAY) or 1
    weights = [FOCUS_WEIGHT / n_focus if word == FOCUS_WORD else 1 for _, word in EVERYDAY]
    for sentence, word in sample_k(rng, EVERYDAY, N_EVERYDAY, weights):
        items.append(("Social History", *context(sentence, word, "everyday")))

    if diagnosis == NO_DISEASE:
        items.append(("Assessment", rng.choice(DIAGNOSIS_NONE), [], []))
    else:
        items.append(("Assessment", *mention(rng, DIAGNOSIS_PRESENT, diagnosis, "disease", "present")))

    others = [d for d in ALL_DISEASES if d != diagnosis]
    weights = [1 + len(DISEASE_SYMPTOMS[d] & set(present)) for d in others]
    for d in sample_k(rng, others, N_RULED_OUT, weights):
        items.append(("Assessment", *mention(rng, DISEASE_RULED_OUT, d, "disease", "absent")))

    return items

def render_markdown(report_id, name, items):
    lines = ["# Zombie Clinical Report", "", f"**Case ID:** {report_id}", f"**Patient:** {name}"]
    for section in SECTIONS:
        sentences = [sentence for sec, sentence, _, _ in items if sec == section]
        if sentences:
            lines += ["", f"## {section}", " ".join(sentences)]
    return "\n".join(lines) + "\n"

In [7]:
REPORTS_DIR.mkdir(exist_ok=True)
for old_file in REPORTS_DIR.glob("*.md"):
    old_file.unlink()

sentence_rows, annotation_rows, context_rows = [], [], []

for index, row in patients.iterrows():
    for variant in range(1, VARIANTS_PER_PATIENT + 1):
        report_id = f"ZH-{index + 1:03d}"
        file_name = slugify(row["Name"])
        if VARIANTS_PER_PATIENT > 1:
            report_id += f"-{variant}"
            file_name += f"-{variant}"

        items = generate_report(row, index + 1, variant)

        for n, (section, sentence, facts, words) in enumerate(items, start=1):
            sentence_id = f"{report_id}-S{n:02d}"
            sentence_rows.append({"report_id": report_id, "section": section,
                                  "sentence_id": sentence_id, "sentence": sentence})
            for fact in facts:
                annotation_rows.append({"report_id": report_id, "sentence_id": sentence_id, **fact})
            for word in words:
                context_rows.append({"report_id": report_id, "sentence_id": sentence_id, **word})

        (REPORTS_DIR / f"{file_name}.md").write_text(render_markdown(report_id, row["Name"], items))

sentences = pd.DataFrame(sentence_rows)
annotations = pd.DataFrame(annotation_rows)
context_words = pd.DataFrame(context_rows, columns=["report_id", "sentence_id", "word", "context"])

sentences.to_csv(SENTENCES_FILE, index=False)
annotations.to_csv(ANNOTATIONS_FILE, index=False)
context_words.to_csv(CONTEXT_WORDS_FILE, index=False)

print(f"Generated {sentences['report_id'].nunique()} reports in {REPORTS_DIR}/")
print(f"Saved {len(sentences)} sentences to {SENTENCES_FILE}")
print(f"Saved {len(annotations)} annotated facts to {ANNOTATIONS_FILE}")
print(f"Saved {len(context_words)} borrowed words to {CONTEXT_WORDS_FILE}")

Generated 19 reports in reports/
Saved 184 sentences to sentences.csv
Saved 113 annotated facts to annotations.csv
Saved 54 borrowed words to context_words.csv


## Save the lexicon (the "field guide")

The lexicon lists every expression a simple algorithm is allowed to know about, with its category.
Notebook 2 builds its dictionaries from this file.

In [8]:
lexicon_rows = (
    [{"surface": s, "canonical": c, "type": "symptom"} for c, ss in SYMPTOM_PHRASES.items() for s in ss]
    + [{"surface": s, "canonical": c, "type": "disease"} for c, ss in DISEASE_PHRASES.items() for s in ss]
    + [{"surface": p, "canonical": p, "type": "pathogen"} for p in PATHOGENS]
    + [{"surface": w, "canonical": w, "type": "negation"} for w in NEGATION_CUES]
    + [{"surface": w, "canonical": w, "type": "terminator"} for w in SCOPE_TERMINATORS]
)
lexicon = pd.DataFrame(lexicon_rows)
lexicon["surface"] = lexicon["surface"].str.lower()
lexicon.to_csv(LEXICON_FILE, index=False)

lexicon.groupby("type").size()

type
disease       12
negation       9
pathogen       2
symptom       18
terminator     2
dtype: int64

## Sanity checks

The generated facts must agree with the source table:

* every symptom listed in the CSV appears as **present** in that patient's report;
* no **absent** symptom belongs to the patient's row;
* the diagnosis appears as **present** (except for `nothing`) and is never ruled out;
* context sentences contain their borrowed word but **no** expression of the lexicon, and a normal
  finding never borrows from a term the patient has.

In [9]:
for index, row in patients.iterrows():
    for report_id, facts in annotations[annotations["report_id"].str.startswith(f"ZH-{index + 1:03d}")].groupby("report_id"):
        symptoms = facts[facts["type"] == "symptom"]
        present = set(symptoms.loc[symptoms["polarity"] == "present", "canonical"])
        absent = set(symptoms.loc[symptoms["polarity"] == "absent", "canonical"])
        assert present == set(row_symptoms(row)), report_id
        assert not absent & set(row_symptoms(row)), report_id

        assessment = facts.merge(sentences, on=["report_id", "sentence_id"])
        assessment = assessment[assessment["section"] == "Assessment"]
        diagnosed = set(assessment.loc[assessment["polarity"] == "present", "canonical"])
        ruled_out = set(assessment.loc[assessment["polarity"] == "absent", "canonical"])
        expected = set() if row["Diagnosis"] == NO_DISEASE else {row["Diagnosis"]}
        assert diagnosed == expected, report_id
        assert row["Diagnosis"] not in ruled_out, report_id

def tokens_of(text):
    return re.findall(r"[a-z0-9]+", text.lower())

TERM_FORMS = set(lexicon.loc[lexicon["type"].isin(["symptom", "disease", "pathogen"]), "surface"])
NORMAL_SOURCE = {sentence: t for t, templates in NORMAL_FINDINGS.items() for sentence, _ in templates}
PRESENT_TERMS = annotations[annotations["polarity"] == "present"].groupby("report_id")["canonical"].agg(set)

for c in context_words.merge(sentences, on=["report_id", "sentence_id"]).itertuples():
    t = tokens_of(c.sentence)
    grams = {" ".join(t[i:i + n]) for n in range(1, 6) for i in range(len(t) - n + 1)}
    assert c.word in t, c.sentence
    assert not grams & TERM_FORMS, c.sentence
    if c.context == "normal finding":
        assert NORMAL_SOURCE[c.sentence] not in PRESENT_TERMS.get(c.report_id, set()), (c.report_id, c.sentence)

print("All checks passed.")

All checks passed.


In [10]:
# Balance of the facts: how many present / absent mentions of each type.
display(pd.crosstab(annotations["type"], annotations["polarity"], margins=True))

# Borrowed words in context sentences.
pd.crosstab(context_words["word"], context_words["context"], margins=True)

polarity,absent,present,All
type,,,
disease,19,25,44
pathogen,0,8,8
symptom,30,31,61
All,49,64,113


context,everyday,normal finding,All
word,,,
anger,1,5,6
bite,0,2,2
chest,3,2,5
control,1,2,3
fight,4,0,4
finger,1,2,3
infection,1,0,1
limb,0,1,1
limbs,0,3,3


In [11]:
# Two example reports.
for path in sorted(REPORTS_DIR.glob("*.md"))[:2]:
    print(path.read_text())

# Zombie Clinical Report

**Case ID:** ZH-015
**Patient:** Blottork Patter

## Presentation
Findings include yellow tongue. The patient presents with trembling on the little finger.

## General Examination
Mild anger when hungry, which is normal.

## Negative Findings
No signs of intense anger episodes. The patient presents without limb loss.

## Laboratory Analysis
Laboratory tests did not identify any pathogen.

## Clinical History
Previous diagnosis of bacterial infectious process.

## Social History
Mother tongue: Zombish.

## Assessment
Findings are consistent with bite related deficit. No evidence of bacterial infection.

# Zombie Clinical Report

**Case ID:** ZH-004
**Patient:** Bouvossam Damme

## Presentation
Findings include member loss. Examination reveals yellowish tongue. Findings include shaking of the pinky finger.

## General Examination
Mild anger when hungry, which is normal. Stethoscope on the chest detects no heartbeat, as expected.

## Negative Findings
No signs of

In [12]:
# The sentences of one report, with the facts they carry.
first = sentences["report_id"].iloc[0]
sentences[sentences["report_id"] == first].merge(
    annotations, on=["report_id", "sentence_id"], how="left"
)[["section", "sentence", "surface", "canonical", "type", "polarity"]]

,section,sentence,surface,canonical,type,polarity
0,Presentation,The patient presents with paralysis.,paralysis,paralysis,symptom,present
1,Presentation,Findings include yellow tinted tongue.,yellow tinted tongue,yellow tongue,symptom,present
2,General Examination,Every member is in its place.,NaN,NaN,NaN,NaN
3,Negative Findings,The patient denies trembling on the little fin...,trembling on the little finger,trembling on the little finger,symptom,absent
4,Laboratory Analysis,Blood analysis identified Cyanobacterium zumbi.,Cyanobacterium zumbi,Cyanobacterium zumbi,pathogen,present
5,Clinical History,The patient denies previous viral infectious p...,viral infectious process,viral infection,disease,absent
6,Social History,The patient watched a pillow fight in the wait...,NaN,NaN,NaN,NaN
7,Assessment,Clinical impression: infection of bacterial or...,infection of bacterial origin,bacterial infection,disease,present
8,Assessment,No evidence of bite deficit.,bite deficit,bite deficit,disease,absent
